In [ ]:
!pip install contractions

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import string
import contractions
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
df = pd.read_csv("/content/IMDB-Dataset.csv",engine='python')
print(df.head())
df = df.rename(columns={"review":"text","sentiment":"label"})
df.info()

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    50000 non-null  object
 1   label   50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


**Preprocessing text data**

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'(\W)\1+', r'\1', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = contractions.fix(text)

    return text

In [ ]:
df["text"] = df["text"].apply(preprocess_text)
print(df["text"].head(10))
print(df["label"].unique())

0    one of the other reviewers has mentioned that ...
1    a wonderful little production br br the filmin...
2    i thought this was a wonderful way to spend ti...
3    basically there is a family where a little boy...
4    petter matteis love in the time of money is a ...
5    probably my alltime favorite movie a story of ...
6    i sure would like to see a resurrection of a u...
7    this show was an amazing fresh innovative idea...
8    encouraged by the positive comments about this...
9    if you like original gut wrenching laughter yo...
Name: text, dtype: object
['positive' 'negative']


In [ ]:
vocab = set()
for sentence in df["text"]:
    vocab.update(sentence.split())

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Build word_to_idx safely
word_to_idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word in vocab:
    word_to_idx[word] = len(word_to_idx)

# ✅ THIS IS THE ONLY CORRECT VOCAB SIZE
vocab_size = len(word_to_idx)

max_length = 1000

train_ds,test_ds = train_test_split(df,random_state=45,test_size=0.2)
train_ds = train_ds.reset_index(drop=True)
test_ds = test_ds.reset_index(drop=True)

def encode_and_pad(text):
    words = text.split()

    encoded_text = [
        word_to_idx.get(word, word_to_idx["<UNK>"])
        for word in words
    ]

    if len(encoded_text) < max_length:
        encoded_text += [word_to_idx["<PAD>"]] * (max_length - len(encoded_text))
    else:
        encoded_text = encoded_text[:max_length]

    return encoded_text

In [ ]:
train_ds["text"] = train_ds['text'].apply(encode_and_pad)
test_ds["text"] = test_ds['text'].apply(encode_and_pad)

**Creating Dataset**

In [ ]:
class MovieDatasetCreator(Dataset):
  def __init__(self,df):
    self.X = df["text"]
    self.y = df["label"]
    self.label_map = {"positive":1,"negative":0}

  def __len__(self):
    return len(self.X)

  def __getitem__(self,idx):
    text = self.X.iloc[idx]
    label = self.label_map[self.y.iloc[idx]]

    text_tensor = torch.tensor(text,dtype=torch.long)
    label_tensor = torch.tensor(label,dtype=torch.long)

    return text_tensor,label_tensor

In [ ]:
class SentimentRNN(nn.Module):
  def __init__(self,vocab_size,embed_size,hidden_size,output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.relu = nn.ReLU()
    self.embedding = nn.Embedding(vocab_size,embed_size,padding_idx=word_to_idx[PAD_TOKEN])
    self.rnn = nn.RNN(input_size=embed_size,hidden_size=hidden_size,batch_first=True,nonlinearity="tanh",num_layers=2)
    self.fc1 = nn.Linear(hidden_size,128)
    self.fc = nn.Linear(128,output_size)


  def forward(self,x):
    x = self.embedding(x)
    h0 = torch.zeros(2, x.size(0), self.hidden_size).to(device)
    output,_  = self.rnn(x,h0)
    output = self.relu(self.fc1(output.mean(dim=1)))
    output = self.fc(output)
    return output

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_size,padding_idx=word_to_idx[PAD_TOKEN])
        self.lstm = nn.LSTM(
            embed_size, hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        output = output.mean(dim=1)
        output = self.dropout(output)
        return self.fc(output)

In [ ]:
def model_train(model,optimizer,loss_function,dataloader):
  model.train()
  train_loss = 0.0
  correct = 0
  total = 0
  for X,y in dataloader:
    optimizer.zero_grad()
    X = X.to(device)
    y = y.to(device)

    output = model(X)
    loss = loss_function(output,y)

    loss.backward()
    optimizer.step()

    train_loss = train_loss + loss.item()
    correct = correct + (output.argmax(dim=1) == y).sum().item()
    total = total + y.size(0)

  accuracy = correct * 100 / total
  avg_train_loss = train_loss / len(dataloader)

  return accuracy , avg_train_loss


In [ ]:
train_ds = MovieDatasetCreator(df=train_ds)
test_ds = MovieDatasetCreator(df=test_ds)


train_DataLoader = DataLoader(
    train_ds,
    shuffle=True,
    batch_size = 32
  )

test_DataLoader = DataLoader(
    test_ds,
    shuffle=False,
    batch_size = 32
)


In [ ]:
embed_size = 128
hidden_size = 256
output_size = 2
model = SentimentRNN(vocab_size,embed_size,hidden_size,output_size)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
epochs = 10

for epoch in range(epochs):
  train_acc , train_loss = model_train(model,optimizer,loss_function,train_DataLoader)

  model.eval()
  with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for X,y in test_DataLoader:
      X = X.to(device)
      y = y.to(device)

      output= model(X)

      loss = loss_function(output,y)

      preds = output.argmax(dim=1)
      correct = correct + (preds == y).sum().item()
      total = total + y.size(0)
      test_loss = test_loss + loss.item()


    test_loss_per_epoch = test_loss / len(test_DataLoader)
    test_acc_per_epoch = correct * 100 / total
  print(f"Epoch : {epoch+1} | Train loss : {train_loss:.4f} | Train accuracy : {train_acc:.2f} | Test loss : {test_loss_per_epoch:.4f} | Test Accuracy : {test_acc_per_epoch:.2f}")

Epoch : 1 | Train loss : 0.6525 | Train accuracy : 61.54 | Test loss : 0.6800 | Test Accuracy : 55.06
Epoch : 2 | Train loss : 0.6873 | Train accuracy : 55.28 | Test loss : 0.6799 | Test Accuracy : 59.61
Epoch : 3 | Train loss : 0.6660 | Train accuracy : 58.49 | Test loss : 0.7330 | Test Accuracy : 51.43
Epoch : 4 | Train loss : 0.6800 | Train accuracy : 55.94 | Test loss : 0.6973 | Test Accuracy : 49.67
Epoch : 5 | Train loss : 0.6576 | Train accuracy : 59.89 | Test loss : 0.6299 | Test Accuracy : 63.78
Epoch : 6 | Train loss : 0.6351 | Train accuracy : 61.98 | Test loss : 0.4161 | Test Accuracy : 81.22
Epoch : 7 | Train loss : 0.5913 | Train accuracy : 70.87 | Test loss : 0.6833 | Test Accuracy : 60.57
Epoch : 8 | Train loss : 0.6262 | Train accuracy : 64.35 | Test loss : 0.4986 | Test Accuracy : 75.95
Epoch : 9 | Train loss : 0.4357 | Train accuracy : 80.53 | Test loss : 0.3881 | Test Accuracy : 83.26
Epoch : 10 | Train loss : 0.4744 | Train accuracy : 77.90 | Test loss : 0.5859 | T

In [ ]:
embed_size = 128
hidden_size = 128
model = SentimentLSTM(vocab_size,embed_size,hidden_size)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
epochs = 10

for epoch in range(epochs):
  train_acc , train_loss = model_train(model,optimizer,loss_function,train_DataLoader)

  model.eval()
  with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for X,y in test_DataLoader:
      X = X.to(device)
      y = y.to(device)

      output= model(X)

      loss = loss_function(output,y)

      preds = output.argmax(dim=1)
      correct = correct + (preds == y).sum().item()
      total = total + y.size(0)
      test_loss = test_loss + loss.item()


    test_loss_per_epoch = test_loss / len(test_DataLoader)
    test_acc_per_epoch = correct * 100 / total
  print(f"Epoch : {epoch+1} | Train loss : {train_loss:.4f} | Train accuracy : {train_acc:.2f} | Test loss : {test_loss_per_epoch:.4f} | Test Accuracy : {test_acc_per_epoch:.2f}")

Epoch : 1 | Train loss : 0.6692 | Train accuracy : 59.70 | Test loss : 0.6740 | Test Accuracy : 58.97
Epoch : 2 | Train loss : 0.6153 | Train accuracy : 67.15 | Test loss : 0.5588 | Test Accuracy : 74.66
Epoch : 3 | Train loss : 0.6025 | Train accuracy : 68.05 | Test loss : 0.4790 | Test Accuracy : 77.73
Epoch : 4 | Train loss : 0.3393 | Train accuracy : 86.39 | Test loss : 0.3092 | Test Accuracy : 87.57
Epoch : 5 | Train loss : 0.2066 | Train accuracy : 92.42 | Test loss : 0.2914 | Test Accuracy : 88.71
Epoch : 6 | Train loss : 0.1310 | Train accuracy : 95.59 | Test loss : 0.3351 | Test Accuracy : 88.22
Epoch : 7 | Train loss : 0.0856 | Train accuracy : 97.21 | Test loss : 0.3854 | Test Accuracy : 88.76
Epoch : 8 | Train loss : 0.0453 | Train accuracy : 98.71 | Test loss : 0.4327 | Test Accuracy : 88.05
Epoch : 9 | Train loss : 0.0303 | Train accuracy : 99.19 | Test loss : 0.5144 | Test Accuracy : 88.37
Epoch : 10 | Train loss : 0.0231 | Train accuracy : 99.39 | Test loss : 0.6280 | T